## AND-102 Task 5: ETL Pipeline — Dataset Merging

## AND-102 Task 5: Merge 1 — License + Installed

In [1]:
import pandas as pd

license_df = pd.read_csv('../data/license.csv')
license_df = license_df[license_df['LICENSESTATUS'].isin(['ACTIVE', 'PENDING_RENEWAL'])]


In [2]:
installed_df = pd.read_json('../data/installed.json')
installed_df = installed_df.rename(columns={'Elevating devices number': 'ElevatingDevicesNumber'})


In [3]:
print(f'license rows (filtered): {len(license_df):,}')
print(f'installed rows:          {len(installed_df):,}')


license rows (filtered): 43,297
installed rows:          46,936


In [4]:
# Join key: ElevatingDevicesNumber (renamed from 'Elevating devices number' in installed.json)
merged_df = license_df.merge(installed_df, on='ElevatingDevicesNumber', how='inner')


In [5]:
lost = len(license_df) - len(merged_df)
print(f'merged rows:  {len(merged_df):,}')
print(f'rows lost:    {lost:,}')
if lost == 0:
    print('No rows lost — every ACTIVE/PENDING_RENEWAL license has a matching entry in installed.json.')
else:
    print(
        f'{lost:,} rows lost because some licensed devices (ACTIVE/PENDING_RENEWAL) '
        'have no matching entry in installed.json — the two datasets were '
        'likely produced at different times and do not have identical device coverage.'
    )


merged rows:  43,297
rows lost:    0
No rows lost — every ACTIVE/PENDING_RENEWAL license has a matching entry in installed.json.


In [6]:
location_sample = merged_df[['ElevatingDevicesNumber', 'LocationoftheElevatingDevice', 'Location of Device']].head(10)
print(location_sample.to_string())


   ElevatingDevicesNumber               LocationoftheElevatingDevice                         Location of Device
0                      10  111 WELLESLEY ST W  TORONTO M7A 1A2 ON CA  111 WELLESLEY ST W  TORONTO M7A 1A2 ON CA
1                    1009       404 MAIN ST  WOODSTOCK N4S 7X5 ON CA       404 MAIN ST  WOODSTOCK N4S 7X5 ON CA
2                   10145     45 SECOND ST E  CORNWALL K6H 1V5 ON CA     45 SECOND ST E  CORNWALL K6H 1V5 ON CA
3                    1018        150 SIMCOE ST  LONDON N6A 4M3 ON CA        150 SIMCOE ST  LONDON N6A 4M3 ON CA
4                    1019        150 SIMCOE ST  LONDON N6A 4M3 ON CA        150 SIMCOE ST  LONDON N6A 4M3 ON CA
5                    1020        150 SIMCOE ST  LONDON N6A 4M3 ON CA        150 SIMCOE ST  LONDON N6A 4M3 ON CA
6                    1021        150 SIMCOE ST  LONDON N6A 4M3 ON CA        150 SIMCOE ST  LONDON N6A 4M3 ON CA
7                    1022        150 SIMCOE ST  LONDON N6A 4M3 ON CA        150 SIMCOE ST  LONDON N6A 4M

In [7]:
# Both address strings end with: ... POSTAL_PART1 POSTAL_PART2 PROVINCE COUNTRY
# Province is reliably the second-to-last whitespace-delimited token.
def extract_province(loc):
    if not isinstance(loc, str):
        return None
    parts = loc.strip().split()
    return parts[-2] if len(parts) >= 2 else None

merged_df['province_license'] = merged_df['LocationoftheElevatingDevice'].apply(extract_province)
merged_df['province_installed'] = merged_df['Location of Device'].apply(extract_province)

print(merged_df[['province_license', 'province_installed']].value_counts().head(10))


province_license  province_installed
ON                ON                    43251
Name: count, dtype: int64


In [8]:
before_location = len(merged_df)
merged_df = merged_df[merged_df['province_license'] == merged_df['province_installed']]

print(f'rows before location filter: {before_location:,}')
print(f'rows after location filter:  {len(merged_df):,}')
print(f'rows removed:                {before_location - len(merged_df):,}')


rows before location filter: 43,297
rows after location filter:  43,251
rows removed:                46


**What was extracted:** The province code (e.g. `ON`) was parsed from the full postal address string present in both the `LocationoftheElevatingDevice` column (license dataset) and the `Location of Device` column (installed dataset). Both columns use the format `STREET  CITY POSTAL_PART1 POSTAL_PART2 PROVINCE COUNTRY`, making the province reliably the second-to-last whitespace-delimited token.

**What "match" means:** After the inner merge on `ElevatingDevicesNumber`, both datasets describe the same physical elevator. A province match confirms that the license record and the installed record for a given device ID agree on the province the device is located in. A mismatch would indicate a data quality issue — the same device number appearing in different provinces across datasets, which is physically impossible.

**Why this filter was applied:** This is a consistency check, not a geographic restriction. It removes any rows where the join key produced a cross-province pairing, which would signal a corrupted or duplicated device ID in the source data rather than a real device. Retaining such rows would introduce noise into any downstream geographic or fleet-level analysis.


In [9]:
print(merged_df['Device Type'].nunique(), 'distinct values in Device Type')
print()
print(merged_df['Device Type'].value_counts().to_string())


10 distinct values in Device Type

Device Type
Passenger Elevator      39975
Freight Elevator         1763
LULA Elevator            1181
Observation Elevator      303
Freight Elevator-P         13
Freight Elevator-E          8
Temporary Elevator          4
Sidewalk Elevator           2
Special Installation        1
Material Lift - ATD         1


In [10]:
device_type_map = {
    'Freight Elevator-P': 'Freight Elevator',
    'Freight Elevator-E': 'Freight Elevator',
    'Material Lift - ATD': 'Other',
    'Special Installation': 'Other',
    'Power Type Manlift': 'Other',
}
merged_df['Device Type'] = merged_df['Device Type'].replace(device_type_map)


In [11]:
print(merged_df['Device Type'].nunique(), 'distinct values after cleaning')
print()
print(merged_df['Device Type'].value_counts().to_string())


7 distinct values after cleaning

Device Type
Passenger Elevator      39975
Freight Elevator         1784
LULA Elevator            1181
Observation Elevator      303
Temporary Elevator          4
Sidewalk Elevator           2
Other                       2


**Column chosen:** `Device Type` from `installed.json` — 10 distinct values in the merged dataset before cleaning (11 in the raw file; `Power Type Manlift` was the only value absent after the license filter and province consistency check).

**Original categories and their problems:**

| Original value | Count | Issue |
|---|---|---|
| `Passenger Elevator` | 39,975 | Clean |
| `Freight Elevator` | 1,763 | Clean |
| `Freight Elevator-P` | 13 | Same device class; `-P` suffix is an undocumented sub-variant |
| `Freight Elevator-E` | 8 | Same device class; `-E` suffix is an undocumented sub-variant |
| `LULA Elevator` | 1,181 | Clean (Limited Use / Limited Application) |
| `Observation Elevator` | 303 | Clean |
| `Sidewalk Elevator` | 2 | Clean |
| `Temporary Elevator` | 4 | Clean |
| `Material Lift - ATD` | 1 | Ambiguous; not an elevator class |
| `Special Installation` | 1 | Ambiguous; no standard meaning |

**How they were consolidated:**

- `Freight Elevator-P` and `Freight Elevator-E` → `Freight Elevator`. The `-P` and `-E` suffixes do not appear in any other dataset or the TSSA device classification scheme and are almost certainly data entry variants of the same category. Collapsing them preserves freight elevator counts without fabricating sub-classes.
- `Material Lift - ATD` and `Special Installation` → `Other`. Each appears exactly once and represents a device type outside the standard elevator classification. Grouping them prevents these single-row categories from distorting any type-level aggregation while keeping the rows in the dataset for completeness.


## AND-102 Task 5: Merge 2 — Adding Alterations

In [12]:
altered_df = pd.read_json('../data/altered.json')
altered_df = altered_df.rename(columns={'Elevating Devices Number': 'ElevatingDevicesNumber'})
print(f'altered rows: {len(altered_df):,}')

altered rows: 31,619


In [13]:
merged_altered_df = merged_df.merge(altered_df, on='ElevatingDevicesNumber', how='left')

In [14]:
before_merge2 = len(merged_df)
print(f'rows before Merge 2: {before_merge2:,}')
print(f'rows after Merge 2:  {len(merged_altered_df):,}')
print(f'row increase:        {len(merged_altered_df) - before_merge2:,}')

rows before Merge 2: 43,251
rows after Merge 2:  52,452
row increase:        9,201


In [15]:
heavy_alteration = (
    merged_altered_df.groupby('ElevatingDevicesNumber')['originating service request number']
    .count()
    .rename('alteration_count')
    .reset_index()
)
heavy = heavy_alteration[heavy_alteration['alteration_count'] >= 5]
total_devices = merged_df['ElevatingDevicesNumber'].nunique()

print(f'elevators with 5+ alteration records: {len(heavy):,}')
print(f'total fleet (post Merge 1):            {total_devices:,}')
print(f'proportion:                            {len(heavy) / total_devices:.1%}')

elevators with 5+ alteration records: 51
total fleet (post Merge 1):            43,251
proportion:                            0.1%


**Why a left merge:** A left merge keeps every row from `merged_df` (the license + installed base) and attaches matching alteration records from `altered.json` where they exist. Elevators that have never been altered would be dropped by an inner merge, which would silently shrink the fleet to only the subset with alteration history — not appropriate when the goal is to characterise the full active fleet. With a left merge, unaltered elevators remain in the dataset with `NaN` in the alteration columns.

**What the row count change means:** If the post-merge row count is higher than the pre-merge count, one or more elevators have multiple alteration records. Each record becomes its own row, so a device with *n* alterations contributes *n* rows. The fleet-level analysis therefore operates on the merged frame grouped by `ElevatingDevicesNumber`, not on raw row counts.

## AND-102 Task 5: Merge 3 — Adding Inspections

In [16]:
inspection_df = pd.read_csv('../data/inspection.csv')
print(f'inspection rows (raw): {len(inspection_df):,}')
print(f'unique elevators:      {inspection_df["ElevatingDevicesNumber"].nunique():,}')
max_per_device = inspection_df.groupby('ElevatingDevicesNumber').size().max()
print(f'relationship:          one-to-many — up to {max_per_device} inspection records per elevator')

inspection rows (raw): 143,181
unique elevators:      40,954
relationship:          one-to-many — up to 24 inspection records per elevator


In [17]:
inspection_df['Latest_INSPECTION_Date'] = pd.to_datetime(inspection_df['Latest_INSPECTION_Date'])
inspection_latest_df = (
    inspection_df.sort_values('Latest_INSPECTION_Date', ascending=False)
    .drop_duplicates(subset='ElevatingDevicesNumber', keep='first')
    .reset_index(drop=True)
)
print(f'inspection rows after deduplication: {len(inspection_latest_df):,}')

inspection rows after deduplication: 40,954


In [18]:
merged_inspection_df = merged_altered_df.merge(inspection_latest_df, on='ElevatingDevicesNumber', how='left')

In [19]:
before_merge3 = len(merged_altered_df)
print(f'rows before Merge 3: {before_merge3:,}')
print(f'rows after Merge 3:  {len(merged_inspection_df):,}')
print(f'row change:          {len(merged_inspection_df) - before_merge3:,}')

rows before Merge 3: 52,452
rows after Merge 3:  52,452
row change:          0


**Why this approach:** The inspection dataset has a one-to-many relationship with elevators — up to 24 inspection records per device across 143,181 rows for 40,954 unique elevators. Merging the raw inspection frame directly onto `merged_altered_df` (which is itself already expanded by alteration records) would produce a row explosion: every alteration row would be duplicated for each inspection record belonging to that elevator, multiplying the frame size by the average inspection count and destroying the one-alteration-per-row structure established in Merge 2.

The chosen approach — deduplicating to the single most recent inspection per elevator before merging — collapses the inspection frame to at most 40,954 rows (one per unique device). After deduplication, the left merge adds exactly one inspection row per device, so the row count of `merged_inspection_df` equals that of `merged_altered_df`. Elevators with no inspection record retain all their alteration rows with `NaN` in the inspection columns.

**Why most recent:** The most recent inspection is the most operationally relevant record — it reflects the current compliance status and condition of the device. Earlier inspections represent superseded assessments and are not needed for fleet-level analysis. `Latest_INSPECTION_Date` was used as the sort key rather than `Earliest_INSPECTION_Date` because it marks when the inspection activity concluded, making it the more meaningful timestamp for recency ranking.

In [20]:
merged_inspection_df.to_csv('../data/merged_elevator_data.csv', index=False)
print(f'rows saved: {len(merged_inspection_df):,}')
print(f'columns ({len(merged_inspection_df.columns)}):')
for col in merged_inspection_df.columns:
    print(f'  {col}')

rows saved: 52,452
columns (38):
  ElevatingDevicesNumber
  LocationoftheElevatingDevice
  ElevatingDevicesLicenseNumber
  LICENSESTATUS
  LICENSEEXPIRYDATE
  LICENSEHOLDER
  LICENSEHOLDERACCOUNTNUMBER
  LICENSEHOLDERADDRESS
  BILLINGCUSTOMER
  BILLINGADDRESS
  BILLINGACCOUNT
  Owner Name
  Owner Address
  Owner Account Number
  Device Class
  Device Type
  DeviceStatus
  Location of Device
  under review
  province_license
  province_installed
  originating service request number
  Alteration Customer
  Summary
  Inspection number
  Alteration  Location
  Alteration Type
  Status of Alteration Request
  Alteration contractor name
  Billing Customer
  originatingservicerequestnumber
  InspectionCustomer
  InspectionNumber
  InspectionLocation
  InspectionType
  Earliest_INSPECTION_Date
  Latest_INSPECTION_Date
  InspectionOutcome
